In [1]:
import os
import shutil

local_path = "/content/processed_frames_v2"
# ⚠️ 주의: 실제 전체 데이터가 있는 드라이브 경로가 맞는지 확인해 주세요.
# (이전에 project23 안에 있다고 하셨는데, 혹시 위치를 옮기셨다면 이 경로가 맞습니다)
drive_path = "/content/drive/MyDrive/processed_frames_v2"

# 1. 덜 복사된 기존 로컬 폴더 강제 삭제
if os.path.exists(local_path):
    print("🗑️ 기존의 불완전한 로컬 폴더를 삭제 중입니다...")
    shutil.rmtree(local_path)
    print("✅ 삭제 완료!")

# 2. 구글 드라이브에서 100% 다시 복사
print("🚀 구글 드라이브에서 전체 데이터를 다시 고속 복사합니다. (약 3~5분 정도 기다려주세요!)")
!cp -r "{drive_path}" /content/
print("🎉 복사 완료! 이제 모든 데이터가 준비되었습니다.")

🚀 구글 드라이브에서 전체 데이터를 다시 고속 복사합니다. (약 3~5분 정도 기다려주세요!)
cp: cannot stat '/content/drive/MyDrive/processed_frames_v2': No such file or directory
🎉 복사 완료! 이제 모든 데이터가 준비되었습니다.


In [2]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import glob
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# 요청하신 구글 드라이브 원본 경로
drive_path = "/content/drive/MyDrive/processed_frames_v2"
# 복사해서 쓸 코랩 로컬 경로 (GPU 병목 방지용)
local_path = "/content/processed_frames_v2"

# 2. 로컬로 데이터 고속 복사
if not os.path.exists(local_path):
    print("🚀 구글 드라이브에서 코랩 로컬로 데이터 고속 복사 중... (수 분 소요)")
    !cp -r "{drive_path}" /content/
    print("✅ 데이터 로컬 복사 완료! I/O 병목 제거 성공.")
else:
    print("✅ 이미 코랩 로컬에 데이터가 준비되어 있습니다.")

# 3. 데이터셋 클래스 정의
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, seq_length=8, transform=None):
        self.seq_length = seq_length
        self.transform = transform
        self.video_folders = []
        self.labels = []

        real_folders = glob.glob(os.path.join(root_dir, "REAL", "*", "*"))
        self.video_folders.extend(real_folders)
        self.labels.extend([0] * len(real_folders))

        fake_folders = glob.glob(os.path.join(root_dir, "FAKE", "*", "*"))
        self.video_folders.extend(fake_folders)
        self.labels.extend([1] * len(fake_folders))

    def __len__(self): return len(self.video_folders)

    def __getitem__(self, idx):
        folder_path = self.video_folders[idx]
        label = self.labels[idx]

        frame_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")),
                             key=lambda x: int(os.path.basename(x).split('_')[1].split('.')[0]))
        frames = []
        for path in frame_paths:
            img = cv2.imread(path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                if self.transform: img = self.transform(img)
                frames.append(img)

        if 0 < len(frames) < self.seq_length:
            frames.extend([frames[-1]] * (self.seq_length - len(frames)))
        elif len(frames) > self.seq_length:
            frames = frames[:self.seq_length]

        if len(frames) == 0: sequence = torch.zeros((self.seq_length, 3, 224, 224))
        else: sequence = torch.stack(frames)

        return sequence, torch.tensor(label, dtype=torch.float32)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. 데이터 로더 준비
full_dataset = DeepfakeDataset(root_dir=local_path, seq_length=8, transform=transform)

train_idx, val_idx = train_test_split(
    list(range(len(full_dataset))), test_size=0.2, random_state=42, stratify=full_dataset.labels
)

train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
val_dataset = torch.utils.data.Subset(full_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"✅ 총 데이터 로드 성공: 학습용 {len(train_dataset)}개 / 검증용 {len(val_dataset)}개")

Mounted at /content/drive
🚀 구글 드라이브에서 코랩 로컬로 데이터 고속 복사 중... (수 분 소요)
✅ 데이터 로컬 복사 완료! I/O 병목 제거 성공.
✅ 총 데이터 로드 성공: 학습용 1608개 / 검증용 402개


In [3]:
import time
import torch.nn as nn
from torchvision import models
from tqdm import tqdm

# 1. GPU 장치 설정 (가장 중요!)
# 여기서 'cuda'라고 출력되어야 정상적으로 GPU를 사용하는 것입니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 현재 사용 중인 디바이스: {device}")

# 2. 모델 구조 정의 (EfficientNet + LSTM)
class DeepfakeDetector(nn.Module):
    def __init__(self, hidden_dim=256, lstm_layers=1):
        super(DeepfakeDetector, self).__init__()
        efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(efficientnet.children())[:-1])

        self.lstm = nn.LSTM(input_size=1280, hidden_size=hidden_dim,
                            num_layers=lstm_layers, batch_first=True, dropout=0.3 if lstm_layers > 1 else 0)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.size()
        x = x.view(batch_size * seq_length, c, h, w)

        features = self.cnn(x)
        features = features.view(features.size(0), -1)
        features = features.view(batch_size, seq_length, -1)

        _, (hidden, _) = self.lstm(features)
        out = self.classifier(hidden[-1])
        return out

# 3. 모델을 GPU로 이동 (.to(device)가 핵심)
model = DeepfakeDetector(hidden_dim=256).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# 4. 학습 루프 (Training Loop)
num_epochs = 10
best_val_loss = float('inf')

print("\n🚀 GPU 가속을 활용한 학습을 시작합니다...\n")

for epoch in range(num_epochs):
    start_time = time.time()

    # [Train]
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for sequences, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        # 데이터도 GPU로 보냄
        sequences = sequences.to(device)
        labels = labels.unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * sequences.size(0)
        preds = torch.sigmoid(outputs) >= 0.5
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    # [Validation]
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for sequences, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            sequences = sequences.to(device)
            labels = labels.unsqueeze(1).to(device)

            outputs = model(sequences)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * sequences.size(0)
            preds = torch.sigmoid(outputs) >= 0.5
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    # 에폭 결과 출력
    elapsed = time.time() - start_time
    print(f"\n[Epoch {epoch+1}/{num_epochs}] 소요 시간: {elapsed:.1f}초")
    print(f"Train Loss: {train_loss/train_total:.4f} | Train Acc: {train_correct/train_total:.4f}")
    print(f"Val Loss:   {val_loss/val_total:.4f} | Val Acc:   {val_correct/val_total:.4f}")

    # 모델 저장 (내 드라이브에 안전하게 저장)
    if (val_loss/val_total) < best_val_loss:
        best_val_loss = (val_loss/val_total)
        # ⚠️ 모델 가중치를 내 구글 드라이브에 저장합니다.
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deepfake_model.pth')
        print("⭐ 성능이 개선되어 모델을 구글 드라이브에 저장했습니다!")
    print("-" * 50)

🔥 현재 사용 중인 디바이스: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 62.0MB/s]



🚀 GPU 가속을 활용한 학습을 시작합니다...



Epoch 1/10 [Val]: 100%|██████████| 51/51 [07:29<00:00,  8.81s/it]



[Epoch 1/10] 소요 시간: 2274.0초
Train Loss: 0.6574 | Train Acc: 0.6281
Val Loss:   0.6063 | Val Acc:   0.6791
⭐ 성능이 개선되어 모델을 구글 드라이브에 저장했습니다!
--------------------------------------------------


...


[Epoch 8/10] 소요 시간: 106.9초
Train Loss: 0.1665 | Train Acc: 0.9384
Val Loss:   0.5501 | Val Acc:   0.8259
--------------------------------------------------


Epoch 9/10 [Val]: 100%|██████████| 51/51 [00:18<00:00,  2.72it/s]



[Epoch 9/10] 소요 시간: 108.1초
Train Loss: 0.1533 | Train Acc: 0.9409
Val Loss:   0.6699 | Val Acc:   0.8060
--------------------------------------------------


Epoch 10/10 [Val]: 100%|██████████| 51/51 [00:19<00:00,  2.65it/s]


[Epoch 10/10] 소요 시간: 104.8초
Train Loss: 0.1146 | Train Acc: 0.9621
Val Loss:   0.7425 | Val Acc:   0.8109
--------------------------------------------------


In [6]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [8]:
# 1. 구글 드라이브 대신 코랩 로컬 디스크에 먼저 피신(저장)시킵니다.
local_save_path = "/content/dfdc_part_0_test_results.csv"
df.to_csv(local_save_path, index=False, encoding='utf-8-sig')

print("✅ 코랩 로컬에 결과를 안전하게 저장했습니다!")

# 2. 화면에 바로 결과 띄워보기
from google.colab import data_table
data_table.enable_dataframe_formatter()
display(df)

✅ 코랩 로컬에 결과를 안전하게 저장했습니다!


,파일명,판정,확률
0,yotyalfqqv.mp4,FAKE,97.7%
1,gwsmeakcic.mp4,ERROR,얼굴 인식 실패
2,gujnhuvarb.mp4,ERROR,얼굴 인식 실패
3,gvcrwbnzpa.mp4,ERROR,얼굴 인식 실패
4,gzkggvloao.mp4,ERROR,얼굴 인식 실패
...,...,...,...
1329,gvlraqlweh.mp4,ERROR,얼굴 인식 실패
1330,gvbcwuyaot.mp4,ERROR,얼굴 인식 실패
1331,guqayoypew.mp4,ERROR,얼굴 인식 실패
1332,gumgvhycum.mp4,ERROR,얼굴 인식 실패


In [9]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # 강제 재연결

# 재연결 후 드라이브로 복사
!cp /content/dfdc_part_0_test_results.csv /content/drive/MyDrive/
print("✅ 구글 드라이브로 복사 완료!")

Mounted at /content/drive
✅ 구글 드라이브로 복사 완료!


In [11]:
import os
import cv2
import glob
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import mediapipe as mp
from torchvision import models, transforms
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm

# =========================================================
# 1. 경로 및 디바이스 설정
# =========================================================
TEST_FOLDER_PATH = "/content/drive/MyDrive/data/dfdc_train_part_0"
MODEL_WEIGHTS_PATH = '/content/drive/MyDrive/best_deepfake_model.pth' # 저장된 모델 경로

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 디바이스 세팅: {device}")

# =========================================================
# 2. MediaPipe 초기화 및 인덱스 설정 (회원님 코드와 동일)
# =========================================================
os.system("wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task")

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    min_face_detection_confidence=0.3
)
detector = vision.FaceLandmarker.create_from_options(options)

CHIN_IDX = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
            397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136,
            172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
LIP_IDX = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
           291, 375, 321, 405, 314, 17, 84, 181, 91, 146]

# =========================================================
# 3. AI 모델 구조 및 가중치 로드
# =========================================================
class DeepfakeDetector(nn.Module):
    def __init__(self, hidden_dim=256, lstm_layers=1):
        super(DeepfakeDetector, self).__init__()
        efficientnet = models.efficientnet_b0(weights=None)
        self.cnn = nn.Sequential(*list(efficientnet.children())[:-1])
        self.lstm = nn.LSTM(input_size=1280, hidden_size=hidden_dim, num_layers=lstm_layers, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.size()
        x = x.view(batch_size * seq_length, c, h, w)
        features = self.cnn(x).view(batch_size * seq_length, -1).view(batch_size, seq_length, -1)
        _, (hidden, _) = self.lstm(features)
        return self.classifier(hidden[-1])

model = DeepfakeDetector(hidden_dim=256).to(device)
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.eval()

# =========================================================
# 4. 실시간 전처리 함수 (회원님의 최적화 로직 100% 반영)
# =========================================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def apply_facial_mask_tensor(frame, img_size=224):
    small = cv2.resize(frame, (320, 180))
    h_orig, w_orig = frame.shape[:2]

    rgb_frame = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    results = detector.detect(mp_image)

    if not results.face_landmarks: return None

    mask = np.zeros((h_orig, w_orig), dtype=np.uint8)
    face_landmarks = results.face_landmarks[0]
    landmarks = [(int(lm.x * w_orig), int(lm.y * h_orig)) for lm in face_landmarks]

    line_thickness = max(2, int(w_orig * 0.015))
    cv2.polylines(mask, [np.array([landmarks[i] for i in CHIN_IDX], np.int32)], False, 255, line_thickness)
    cv2.polylines(mask, [np.array([landmarks[i] for i in LIP_IDX], np.int32)], True, 255, line_thickness)

    masked_frame = cv2.bitwise_and(frame, frame, mask=mask)
    all_pts = np.array(landmarks)
    x_min, y_min = np.min(all_pts, axis=0)
    x_max, y_max = np.max(all_pts, axis=0)

    margin_x, margin_y = int((x_max - x_min) * 0.2), int((y_max - y_min) * 0.2)
    x_min, x_max = max(0, x_min - margin_x), min(w_orig, x_max + margin_x)
    y_min, y_max = max(0, y_min - margin_y), min(h_orig, y_max + margin_y)

    cropped = masked_frame[y_min:y_max, x_min:x_max]
    if cropped.size != 0:
        return transform(cv2.cvtColor(cv2.resize(cropped, (img_size, img_size)), cv2.COLOR_BGR2RGB))
    return None

def extract_and_process(cap, fps, start_sec, duration_sec=3, num_frames=8):
    start_frame = int(fps * start_sec)
    max_frames = int(fps * duration_sec)
    interval = max(1, max_frames // num_frames)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    tensors = []
    idx = 0
    while idx < max_frames and len(tensors) < num_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % interval == 0:
            tensor = apply_facial_mask_tensor(frame)
            if tensor is not None:
                tensors.append(tensor)
        idx += 1
    return tensors

def process_single_video(video_path, seq_length=8):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30
    total_sec = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps

    # 1차 시도: 중간 3초
    mid_start = max(0, (total_sec / 2) - 1.5)
    tensors = extract_and_process(cap, fps, start_sec=mid_start)

    # 2차 시도 (Fallback): 앞 3초
    if len(tensors) == 0:
        tensors = extract_and_process(cap, fps, start_sec=0)

    cap.release()

    if len(tensors) == 0: return None

    # 패딩 및 자르기
    if len(tensors) < seq_length:
        tensors.extend([tensors[-1]] * (seq_length - len(tensors)))
    elif len(tensors) > seq_length:
        tensors = tensors[:seq_length]

    return torch.stack(tensors).unsqueeze(0) # [1, 8, 3, 224, 224]

# =========================================================
# 5. 폴더 일괄 분석 및 CSV 저장
# =========================================================
video_files = glob.glob(os.path.join(TEST_FOLDER_PATH, "*.mp4"))

if len(video_files) == 0:
    print("❌ 해당 폴더에 mp4 파일이 없습니다. 경로를 확인해주세요.")
else:
    print(f"총 {len(video_files)}개의 영상을 발견했습니다. 본격적인 검증을 시작합니다!\n")
    results = []

    for video_path in tqdm(video_files, desc="영상 분석 중"):
        filename = os.path.basename(video_path)
        input_tensor = process_single_video(video_path)

        if input_tensor is None:
            results.append({"파일명": filename, "판정": "ERROR", "확률": "얼굴 인식 실패"})
            continue

        with torch.no_grad():
            output = model(input_tensor.to(device))
            prob = torch.sigmoid(output).item()

        if prob >= 0.5:
            results.append({"파일명": filename, "판정": "FAKE", "확률": f"{prob*100:.1f}%"})
        else:
            results.append({"파일명": filename, "판정": "REAL", "확률": f"{(1-prob)*100:.1f}%"})

    df = pd.DataFrame(results)
    csv_save_path = "/content/drive/MyDrive/dfdc_part_0_test_results.csv"
    df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')

    print("\n" + "="*50)
    print("🎯 분석 결과 샘플 (첫 10개)")
    print("="*50)
    print(df.head(10).to_string(index=False))
    print("="*50)
    print(f"🎉 엑셀(CSV) 파일이 드라이브에 저장되었습니다: {csv_save_path}")

🔥 디바이스 세팅: cuda
총 1334개의 영상을 발견했습니다. 본격적인 검증을 시작합니다!



영상 분석 중: 100%|██████████| 1334/1334 [1:22:47<00:00,  3.72s/it]


🎯 분석 결과 샘플 (첫 10개)
           파일명    판정       확률
yotyalfqqv.mp4  FAKE    97.7%
gwsmeakcic.mp4 ERROR 얼굴 인식 실패
gujnhuvarb.mp4 ERROR 얼굴 인식 실패
gvcrwbnzpa.mp4 ERROR 얼굴 인식 실패
gzkggvloao.mp4 ERROR 얼굴 인식 실패
gxonqiyzpk.mp4 ERROR 얼굴 인식 실패
gxzdueambi.mp4 ERROR 얼굴 인식 실패
gyosvoeamf.mp4 ERROR 얼굴 인식 실패
gwzfugbidx.mp4 ERROR 얼굴 인식 실패
haxnhfvqvs.mp4 ERROR 얼굴 인식 실패
🎉 엑셀(CSV) 파일이 드라이브에 저장되었습니다: /content/drive/MyDrive/dfdc_part_0_test_results.csv


In [12]:
import json
import pandas as pd
import os

# =========================================================
# 1. 경로 설정 (파일이 있는 정확한 경로)
# =========================================================
CSV_PATH = "/content/drive/MyDrive/dfdc_part_0_test_results.csv"
JSON_PATH = "/content/drive/MyDrive/data/dfdc_train_part_0/metadata.json"

if not os.path.exists(CSV_PATH) or not os.path.exists(JSON_PATH):
    print("❌ 파일을 찾을 수 없습니다. 경로를 다시 한번 확인해 주세요.")
else:
    # =========================================================
    # 2. 데이터 불러오기
    # =========================================================
    # 정답지 로드
    with open(JSON_PATH, 'r') as f:
        ground_truth = json.load(f)

    # AI 답안지(CSV) 로드
    df = pd.read_csv(CSV_PATH)

    # 총 영상 수 확인
    total_videos = len(df)

    # 'ERROR'가 아닌 정상 판정된 데이터만 필터링
    valid_df = df[df['판정'] != 'ERROR']
    detected_count = len(valid_df)
    error_count = total_videos - detected_count

    # =========================================================
    # 3. 채점 진행
    # =========================================================
    correct_total = 0

    real_total, real_correct = 0, 0
    fake_total, fake_correct = 0, 0

    for index, row in valid_df.iterrows():
        filename = row['파일명']
        ai_pred = row['판정']

        # 정답지에 해당 파일이 있는지 확인
        if filename in ground_truth:
            actual_label = ground_truth[filename]['label']

            # 카운트 누적
            if actual_label == 'REAL':
                real_total += 1
                if ai_pred == 'REAL':
                    real_correct += 1
                    correct_total += 1
            elif actual_label == 'FAKE':
                fake_total += 1
                if ai_pred == 'FAKE':
                    fake_correct += 1
                    correct_total += 1

    # =========================================================
    # 4. 결과 출력
    # =========================================================
    accuracy = (correct_total / detected_count) * 100 if detected_count > 0 else 0
    real_acc = (real_correct / real_total) * 100 if real_total > 0 else 0
    fake_acc = (fake_correct / fake_total) * 100 if fake_total > 0 else 0

    print("="*50)
    print("🎯 딥페이크 AI 최종 실전 채점 결과")
    print("="*50)
    print(f"총 검사한 영상 수 : {total_videos}개")
    print(f" - 👤 얼굴 인식 성공 : {detected_count}개")
    print(f" - ❌ 얼굴 인식 실패 : {error_count}개 (채점 제외)")
    print("-" * 50)
    print(f"✅ 최종 정확도 (Accuracy) : {accuracy:.2f}% ({correct_total}/{detected_count})")
    print("-" * 50)
    print("🔍 세부 분석")
    print(f" - 진짜(REAL) 영상 정답률 : {real_acc:.2f}% ({real_correct}/{real_total})")
    print(f" - 가짜(FAKE) 영상 정답률 : {fake_acc:.2f}% ({fake_correct}/{fake_total})")
    print("="*50)

🎯 딥페이크 AI 최종 실전 채점 결과
총 검사한 영상 수 : 1334개
 - 👤 얼굴 인식 성공 : 332개
 - ❌ 얼굴 인식 실패 : 1002개 (채점 제외)
--------------------------------------------------
✅ 최종 정확도 (Accuracy) : 80.72% (268/332)
--------------------------------------------------
🔍 세부 분석
 - 진짜(REAL) 영상 정답률 : 16.67% (4/24)
 - 가짜(FAKE) 영상 정답률 : 85.71% (264/308)
